# Аналитика потребления контента стримингового сервиса (PySpark)**Задача.** Сервис аудиокниг, книг и комиксов ежедневно фиксирует около 10 тысяч событий прослушивания. Продуктовой команде нужны регулярные метрики потребления — по типам контента, географии и сегментам аудитории — без ручных выгрузок и с возможностью пересчитать историю.**Цель ноутбука.** Понять структуру, объём и качество сырых логов и на основе этого решить: какие витрины строить, с какой гранулярностью их обновлять и какие проверки качества поставить перед загрузкой в хранилище. Реализация пайплайна — в `src/` и `dags/`.

**Данные**- `content` — справочник произведений (книги / аудиокниги / комиксы): id, тип, название, длительность, темы, автор.- `audition` — журнал событий прослушивания: пользователь, платформа, бизнес-дата, длительность сессии, флаги adult/kids, гео, ссылка на контент.

## 1. Загрузка данных и знакомство

In [ ]:
from pyspark.sql import SparkSessionfrom pyspark.sql import functions as Ffrom pyspark.sql.types import (    StructType, StructField, StringType, IntegerType, DoubleType, BooleanType,)spark = SparkSession.builder.appName("streaming-content").getOrCreate()

In [ ]:
# Явные схемы — на проде не полагаемся на inferSchemacontent_schema = StructType([    StructField("main_content_id", StringType()),    StructField("main_content_type", StringType()),    StructField("main_content_name", StringType()),    StructField("main_content_duration_hours", DoubleType()),    StructField("published_topic_title_list", StringType()),    StructField("main_author_id", StringType()),])audition_schema = StructType([    StructField("usage_geo_id", IntegerType()),    StructField("audition_id", IntegerType()),    StructField("puid", StringType()),    StructField("usage_platform_ru", StringType()),    StructField("msk_business_dt_str", StringType()),    StructField("app_version", StringType()),    StructField("adult_content_flg", BooleanType()),    StructField("hours", DoubleType()),    StructField("hours_sessions_long", DoubleType()),    StructField("kids_content_flg", BooleanType()),    StructField("main_content_id", StringType()),    StructField("usage_geo_id_name", StringType()),    StructField("usage_country_name", StringType()),])content_df = spark.read.csv("data/content.csv", schema=content_schema, header=False)audition_df = spark.read.csv("data/audition.csv", schema=audition_schema, header=False)

In [ ]:
content_df.show(5)audition_df.show(5)

In [ ]:
print("content:", content_df.count(), "строк")print("audition:", audition_df.count(), "строк")

**Наблюдения.** `content` — медленно меняющийся справочник произведений (~31,7 тыс.); `audition` — журнал событий (~1 млн строк). Поле `msk_business_dt_str` загружено строкой — приведём к дате; в `app_version` есть пропуски.

## 2. Предобработка и производные признаки

In [ ]:
# Дата из строки, минуты сессии, признак выходного дняaudition_df = (    audition_df    .withColumn("business_dt", F.to_date("msk_business_dt_str", "yyyy-MM-dd"))    .withColumn("minutes", (F.col("hours_sessions_long") * 60).cast(IntegerType()))    .withColumn("is_weekend", F.dayofweek("business_dt").isin(1, 7)))# Убираем пропуски в ключевом флагеaudition_df = audition_df.filter(F.col("adult_content_flg").isNotNull())

## 3. Период и гранулярность данныхКлючевой вопрос для архитектуры пайплайна: данные приходят разовым срезом или посуточно?

In [ ]:
period = audition_df.agg(    F.countDistinct("business_dt").alias("days"),    F.min("business_dt").alias("from_dt"),    F.max("business_dt").alias("to_dt"),)period.show()

In [ ]:
# Сколько событий приходится на день(audition_df.groupBy("business_dt")    .agg(F.count("*").alias("events"))    .orderBy("business_dt").show(5))

**Вывод — и он определяет дизайн пайплайна.**События покрывают **102 дня** (2024-09-01 … 2024-12-11), примерно по **7–12 тыс. в день**. Это не разовый срез, а суточный поток. Отсюда решения:- обрабатывать данные **по одной бизнес-дате за запуск** — не пересчитывать всю историю каждый раз;- партиционировать витрины по `business_dt`, чтобы перезалив одной даты не трогал остальные;- сделать загрузку **идемпотентной**: повторный запуск за дату заменяет её партицию, а не дублирует данные.

## 4. Потребление: будни/выходные и тип аудитории

In [ ]:
(audition_df.groupBy("is_weekend")    .agg(F.sum("minutes").alias("total_minutes"))    .orderBy("is_weekend").show())

In [ ]:
(audition_df.groupBy("adult_content_flg", "is_weekend")    .agg(F.sum("minutes").alias("total_minutes"))    .orderBy("adult_content_flg", "is_weekend").show())

**Выводы.**- Потребление в будни заметно выше, чем в выходные, — паттерн «фонового» сервиса под ежедневную рутину (дорога, работа, перерывы).- Взрослый контент за период потребляется примерно в 4 раза больше детского по суммарным минутам — основная аудитория взрослая.- Детский контент в будни слушают заметно больше, чем в выходные (вероятно, по дороге в сад/школу).Значит, разрезы «тип контента × adult/kids × будни-выходные» имеют смысл — выносим их в отдельную витрину.

## 5. Соединение со справочником контента

In [ ]:
joined_df = audition_df.join(content_df, on="main_content_id", how="inner")print("до join:", audition_df.count())print("после join:", joined_df.count())

In [ ]:
print("уникальных пользователей:", joined_df.select("puid").distinct().count())joined_df.select("main_content_type").distinct().show()

**Выводы.**- При `inner join` теряется ~0,6% событий (контент отсутствует в справочнике: удалён из каталога, ещё не загружен или технический id) — потеря некритична.- Аудитория почти не страдает: <0,2% пользователей слушали только «потерянный» контент.- Форматы контента: Audiobook, Book, Comicbook — данные корректно категоризированы.- `audition_id` уникален и годится как ключ дедупликации.

## 6. Итоги: что из разведки попало в пайплайн**Витрины (все с `business_dt`):**- `mart_daily_content_type` — минуты / сессии / уникальные пользователи по типу контента, флагам adult-kids и признаку выходного дня;- `mart_geo` — потребление по стране и региону;- `mart_content_top` — топ произведений по минутам.**Обновление:** инкрементально, по одной бизнес-дате; партиции по `business_dt`; загрузка идемпотентна (перезалив даты не даёт дублей), поэтому ретраи и пересчёт истории безопасны.**Проверки качества перед загрузкой:** партиция за дату непуста; нет отрицательных минут, сессий и пользователей; ключ витрины уникален.Реализация — `src/` и `dags/`; витрины ложатся в ClickHouse, поверх них дашборд в Superset.